# 📊 [Baseline] 샤프 지수 최적화 파이프라인 (연도별 무위험 이자율 반영)
본 노트북은 Lending Club 데이터를 활용하여 부도 여부를 예측하는 7가지 베이스라인 모델을 학습합니다.
- **데이터 분할:** Train과 Validation을 3:1(75%:25%)로 분할합니다.
- **불균형 처리:** Train 셋에만 무작위 언더샘플링(Random Undersampling)을 적용하여 1:1 비율을 맞춥니다.
- **샤프 지수 계산:** 대출 발행 연도(`issue_year`)별 미국 3년 만기 국채 평균 수익률을 매핑하여, 거시경제 상황이 반영된 정교한 **초과 수익률(Excess Return)**과 **샤프 지수(Sharpe Ratio)**를 산출합니다.

In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
import warnings
warnings.filterwarnings('ignore')

print("=== 1. 데이터 로드 및 기본 전처리 ===")
file_path = '/Users/hantaeho/Documents/statistic_science/stat_Assignment/lending_club_2020_train.csv'
df = pd.read_csv(file_path, low_memory=False)

# 확정 대출 및 개인 대출 필터링
valid_statuses = ['Fully Paid', 'Charged Off', 'Default']
df_base = df[df['loan_status'].isin(valid_statuses)].copy()
if 'application_type' in df_base.columns:
    df_base = df_base[df_base['application_type'] == 'Individual'].copy()

# 타겟 변수 설정 (부도=1, 정상=0)
df_base['default'] = np.where(df_base['loan_status'].isin(['Charged Off', 'Default']), 1, 0)

# 날짜형 변환 및 대출 발행 연도 추출
for col in ['issue_d', 'earliest_cr_line', 'last_pymnt_d']:
    if col in df_base.columns:
        df_base[col] = pd.to_datetime(df_base[col], format='%b-%Y', errors='coerce')

df_base['issue_year'] = df_base['issue_d'].dt.year

# 💡 미국 3년 만기 국채(Treasury Yield) 연도별 평균 데이터 매핑
treasury_rates = {
    2007: 0.044, 2008: 0.022, 2009: 0.014, 2010: 0.011,
    2011: 0.008, 2012: 0.004, 2013: 0.007, 2014: 0.015,
    2015: 0.013, 2016: 0.010, 2017: 0.016, 2018: 0.026,
    2019: 0.019, 2020: 0.004
}
# 매핑되지 않은 연도는 전체 평균(약 1.5%)으로 결측치 처리
df_base['risk_free_rate'] = df_base['issue_year'].map(treasury_rates).fillna(0.015)

print(f"데이터 로드 및 국채 수익률 매핑 완료. 형태: {df_base.shape}")

=== 1. 데이터 로드 및 기본 전처리 ===
데이터 로드 및 국채 수익률 매핑 완료. 형태: (1074490, 144)


## Step 1. 파생변수 생성 및 결측치 지시자 적용
내생/사후 변수를 독립 변수에서 철저히 제외하고, 비율 기반 파생변수 및 결측치 지시자(Missing Indicator)를 생성하여 데이터의 정보량을 극대화합니다.

In [25]:
print("=== 2. 변수 정제 및 결측치 고도화 ===")

# ──────────────────────────────────────────────
# ★ 평가용 컬럼을 가장 먼저 보관 (제거 전에!)
# ──────────────────────────────────────────────
eval_cols = {}
for col in ['total_pymnt', 'funded_amnt', 'loan_status', 'default', 'issue_d', 'last_pymnt_d', 'earliest_cr_line', 'term']:
    if col in df_base.columns:
        eval_cols[col] = df_base[col].copy()
df_eval = pd.DataFrame(eval_cols, index=df_base.index)
print(f"평가용 컬럼 별도 보관: {list(df_eval.columns)}")

# ──────────────────────────────────────────────
# 1. 내생변수 + 사후변수 + 식별자 + 상수 차단
# ──────────────────────────────────────────────
exclude_cols = [
    'int_rate', 'grade', 'sub_grade', 'installment',
    'funded_amnt', 'funded_amnt_inv', 'initial_list_status',
    'out_prncp', 'out_prncp_inv',
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
    'recoveries', 'collection_recovery_fee',
    'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d',
    'last_fico_range_low', 'last_fico_range_high',
    'hardship_flag', 'hardship_type', 'hardship_reason', 'hardship_status',
    'hardship_start_date', 'hardship_end_date', 'hardship_amount',
    'hardship_length', 'hardship_dpd', 'hardship_loan_status',
    'hardship_payoff_balance_amount', 'hardship_last_payment_amount',
    'deferral_term', 'payment_plan_start_date',
    'orig_projected_additional_accrued_interest',
    'debt_settlement_flag',
    'id', 'member_id', 'url', 'desc',
    'emp_title', 'title',
    'policy_code', 'pymnt_plan',
]
df_base.drop(columns=[c for c in exclude_cols if c in df_base.columns], inplace=True)

# ──────────────────────────────────────────────
# 2. 문자열 → 숫자 변환 (이미 변환됐으면 건너뜀)
# ──────────────────────────────────────────────
if 'term' in df_base.columns and df_base['term'].dtype == 'object':
    df_base['term'] = df_base['term'].str.extract(r'(\d+)').astype(float)

if 'emp_length' in df_base.columns and df_base['emp_length'].dtype == 'object':
    emp_map = {'< 1 year':0, '1 year':1, '2 years':2, '3 years':3, '4 years':4,
               '5 years':5, '6 years':6, '7 years':7, '8 years':8, '9 years':9, '10+ years':10}
    df_base['emp_length'] = df_base['emp_length'].map(emp_map)

if 'revol_util' in df_base.columns and df_base['revol_util'].dtype == 'object':
    df_base['revol_util'] = pd.to_numeric(
        df_base['revol_util'].astype(str).str.replace('%', '', regex=False).str.strip(),
        errors='coerce'
    )

# ──────────────────────────────────────────────
# 3. 결측치 40% 이상 열 제거
# ──────────────────────────────────────────────
missing_ratio = df_base.isnull().sum() / len(df_base)
df_base = df_base[missing_ratio[missing_ratio < 0.4].index].copy()

# ──────────────────────────────────────────────
# 4. 경제적 파생변수
# ──────────────────────────────────────────────
if 'fico_range_high' in df_base.columns and 'fico_range_low' in df_base.columns:
    df_base['fico_range_avg'] = (df_base['fico_range_high'] + df_base['fico_range_low']) / 2

if 'num_bc_sats' in df_base.columns and 'num_bc_tl' in df_base.columns:
    df_base['ratio_satis'] = df_base['num_bc_sats'] / df_base['num_bc_tl']
    df_base['ratio_satis'].fillna(0, inplace=True)

if 'num_op_rev_tl' in df_base.columns and 'num_rev_accts' in df_base.columns:
    df_base['ratio_open_revolving'] = df_base['num_op_rev_tl'] / df_base['num_rev_accts']
    df_base['ratio_open_revolving'].fillna(0, inplace=True)

if 'issue_d' in df_base.columns and 'earliest_cr_line' in df_base.columns:
    df_base['credit_history_days'] = (df_base['issue_d'] - df_base['earliest_cr_line']).dt.days

# ──────────────────────────────────────────────
# 5. 결측 더미 (정보성 결측)
# ──────────────────────────────────────────────
missing_info_cols = ['revol_util', 'pct_tl_nvr_dlq', 'mths_since_recent_bc',
                     'percent_bc_gt_75', 'mo_sin_old_rev_tl_op', 'inq_last_6mths']
for col in missing_info_cols:
    if col in df_base.columns:
        df_base[col] = pd.to_numeric(df_base[col], errors='coerce')
        if df_base[col].isnull().sum() > 0:
            df_base[f'{col}_is_missing'] = np.where(df_base[col].isnull(), 1, 0)
            df_base[col].fillna(df_base[col].mean(), inplace=True)

# ──────────────────────────────────────────────
# 6. 범주형 → 더미 변환
# ──────────────────────────────────────────────
high_card_cols = ['zip_code', 'addr_state']
df_base.drop(columns=[c for c in high_card_cols if c in df_base.columns], inplace=True)

if 'purpose' in df_base.columns:
    df_base['purpose'] = df_base['purpose'].map(
        {'debt_consolidation': 'Financial', 'credit_card': 'Financial'}
    ).fillna('Other')

if 'home_ownership' in df_base.columns:
    df_base = df_base[df_base['home_ownership'].isin(['MORTGAGE', 'RENT', 'OWN'])]

if 'application_type' in df_base.columns:
    df_base.drop(columns=['application_type'], inplace=True)

dummy_cols = ['purpose', 'home_ownership', 'verification_status']
df_base = pd.get_dummies(df_base, columns=[c for c in dummy_cols if c in df_base.columns])

# ──────────────────────────────────────────────
# 7. 잔여 문자열/날짜 컬럼 최종 제거
# ──────────────────────────────────────────────
obj_cols = df_base.select_dtypes(include=['object', 'datetime64']).columns.tolist()
if obj_cols:
    print(f"⚠️ 모델에 넣을 수 없는 잔여 컬럼 제거: {obj_cols}")
    df_base.drop(columns=obj_cols, inplace=True)

# ──────────────────────────────────────────────
# 8. 잔여 수치형 결측 → 평균 대치
# ──────────────────────────────────────────────
numeric_cols = df_base.select_dtypes(include=[np.number]).columns
df_base[numeric_cols] = df_base[numeric_cols].fillna(df_base[numeric_cols].mean())

# df_eval 인덱스도 df_base에 맞추기 (home_ownership 필터로 행이 줄었을 수 있음)
df_eval = df_eval.loc[df_base.index]

# ──────────────────────────────────────────────
# 9. 최종 점검
# ──────────────────────────────────────────────
print(f"\n전처리 완료: {df_base.shape[0]:,}행 × {df_base.shape[1]}열")
print(f"남은 object 컬럼: {df_base.select_dtypes(include='object').columns.tolist()}")
print(f"남은 NaN 수: {df_base.isnull().sum().sum()}")
print(f"df_eval 보관 컬럼: {list(df_eval.columns)}")
print(f"df_eval funded_amnt 존재: {'funded_amnt' in df_eval.columns}")

=== 2. 변수 정제 및 결측치 고도화 ===
평가용 컬럼 별도 보관: ['total_pymnt', 'funded_amnt', 'loan_status', 'default', 'issue_d', 'last_pymnt_d', 'earliest_cr_line', 'term']
⚠️ 모델에 넣을 수 없는 잔여 컬럼 제거: ['term', 'emp_length', 'issue_d', 'loan_status', 'earliest_cr_line', 'last_pymnt_d']

전처리 완료: 1,073,657행 × 74열
남은 object 컬럼: []
남은 NaN 수: 1073657
df_eval 보관 컬럼: ['total_pymnt', 'funded_amnt', 'loan_status', 'default', 'issue_d', 'last_pymnt_d', 'earliest_cr_line', 'term']
df_eval funded_amnt 존재: True


## Step 2. 3:1 Train/Validation 분할 및 언더샘플링
평가 시 실제 수익률 계산을 위해 상환 기간(`pymnt_term_years`)을 미리 확보합니다.
Validation 셋은 현실 비율(불균형 상태)을 유지하기 위해 먼저 25%를 떼어내고, **Train 셋 내에서만 무작위 언더샘플링**을 진행하여 1:1로 맞춥니다.

In [26]:
print("=== 3. 데이터 분할 및 랜덤 언더샘플링 ===")

# df_eval에서 평가용 변수를 가져옴 (앞 셀에서 삭제 전에 보관해둔 것)
df_base['total_pymnt'] = df_eval['total_pymnt']
df_base['default'] = df_eval['default']
df_base['issue_d'] = df_eval['issue_d']
df_base['last_pymnt_d'] = df_eval['last_pymnt_d']

# 상환 기간 산출
df_base['pymnt_term_years'] = (df_base['last_pymnt_d'] - df_base['issue_d']).dt.days / 365.25
df_base = df_base[df_base['pymnt_term_years'] > 0].copy()

# df_eval도 같은 인덱스로 맞추기
df_eval = df_eval.loc[df_base.index]

# 독립 변수(X)와 타겟 변수(y) 분리
#   평가용 사후 변수는 X에서 철저히 배제
leakage_cols = ['total_pymnt', 'last_pymnt_d', 'pymnt_term_years',
                'loan_status', 'default', 'issue_d', 'earliest_cr_line']
features = [c for c in df_base.columns if c not in leakage_cols
            and df_base[c].dtype not in ['object', 'datetime64[ns]']]

X = df_base[features]
y = df_base['default']

print(f"설명변수 수: {X.shape[1]}개")
print(f"남은 object/datetime: {X.select_dtypes(include=['object','datetime64']).columns.tolist()}")

# 6:2:2 분할 (Train 60%, Valid 20%, Test 20%)
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=y_tmp
)

# 훈련 데이터 언더샘플링
train_data = pd.concat([X_train, y_train], axis=1)
df_majority = train_data[train_data['default'] == 0]
df_minority = train_data[train_data['default'] == 1]
df_majority_down = resample(df_majority, replace=False,
                            n_samples=len(df_minority), random_state=42)
train_downsampled = pd.concat([df_majority_down, df_minority]).sample(frac=1, random_state=42)

X_train_under = train_downsampled.drop('default', axis=1)
y_train_under = train_downsampled['default']

print(f"\nTrain: {len(X_train):,} | Valid: {len(X_val):,} | Test: {len(X_test):,}")
print(f"언더샘플링 후 Train → 정상: {len(df_majority_down):,} | 부도: {len(df_minority):,} (1:1)")

=== 3. 데이터 분할 및 랜덤 언더샘플링 ===
설명변수 수: 72개
남은 object/datetime: []

Train: 638,835 | Valid: 212,945 | Test: 212,945
언더샘플링 후 Train → 정상: 123,068 | 부도: 123,068 (1:1)


## Step 3. 7종 머신러닝 알고리즘 학습 및 샤프 지수(Sharpe Ratio) 산출
7개의 다양한 분류 모델을 학습시킵니다.
모델이 검증 셋(Validation Set)에서 '정상 상환'으로 판단한 대출 건들에 한해 포트폴리오를 구성하고, **대출 건별 초과 수익률(개별 수익률 - 발행 연도별 국채 수익률)**의 평균과 표준편차를 통해 최종 **샤프 지수**를 도출합니다.

In [31]:
# 모든 컬럼 숫자 강제 변환 + NaN/inf 제거
X_train_under = X_train_under.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
X_val = X_val.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
X_test = X_test.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)

print(f"X_train_under: {X_train_under.shape}, NaN: {X_train_under.isna().sum().sum()}")
print(f"dtypes: {X_train_under.dtypes.value_counts().to_dict()}")



print("=== 4. 7개 모델 학습 및 샤프 비율 평가 ===")

# ──────────────────────────────────────────────
# 미국 국채 수익률 매칭 (3년물 / 5년물, 연도별 평균)
# 출처: U.S. Treasury / FRED
# ──────────────────────────────────────────────
treasury_3y = {
    2007: 4.35, 2008: 2.24, 2009: 1.43, 2010: 1.37, 2011: 0.75,
    2012: 0.36, 2013: 0.76, 2014: 0.97, 2015: 1.16, 2016: 1.01,
    2017: 1.62, 2018: 2.64, 2019: 1.62, 2020: 0.37,
}
treasury_5y = {
    2007: 4.43, 2008: 2.80, 2009: 2.20, 2010: 1.93, 2011: 1.52,
    2012: 0.76, 2013: 1.31, 2014: 1.64, 2015: 1.53, 2016: 1.32,
    2017: 1.92, 2018: 2.73, 2019: 1.69, 2020: 0.53,
}

def get_rf(issue_year, term_months):
    if term_months == 60:
        return treasury_5y.get(issue_year, 1.5) / 100
    else:
        return treasury_3y.get(issue_year, 1.0) / 100

# ──────────────────────────────────────────────
# 샤프 비율 계산 함수
# ──────────────────────────────────────────────
def calc_sharpe(returns, rf_rates):
    excess = returns - rf_rates
    if len(excess) < 30 or np.std(returns) == 0:
        return np.nan
    return np.mean(excess) / np.std(returns)

# ──────────────────────────────────────────────
# Validation 셋 수익률 계산 (df_eval에서 가져옴)
# ──────────────────────────────────────────────
val_eval = df_eval.loc[X_val.index].copy()
val_eval['issue_year'] = val_eval['issue_d'].dt.year
val_eval['rf'] = val_eval.apply(lambda row: get_rf(row['issue_year'], row['term']), axis=1)

F_val = val_eval['funded_amnt'].values
P_val = val_eval['total_pymnt'].values
M_val = pd.to_numeric(val_eval['term'].astype(str).str.extract(r'(\d+)')[0], errors='coerce').values
r_val = (P_val / F_val) ** (12 / M_val) - 1
rf_val = val_eval['rf'].values            

# 벤치마크
all_sharpe = calc_sharpe(r_val, rf_val)
print(f"[벤치마크] 전부 대출 시 샤프: {all_sharpe:.4f}")
print(f"국채 금리 범위: {rf_val.min()*100:.2f}% ~ {rf_val.max()*100:.2f}%\n")

# ──────────────────────────────────────────────
# 7가지 모델 학습 + 임계값 탐색 + 샤프 평가
# ──────────────────────────────────────────────
models_7 = {
    'LightGBM': LGBMClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1, verbose=-1),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1, eval_metric='logloss', verbosity=0),
    'CatBoost': CatBoostClassifier(iterations=100, depth=5, random_state=42, verbose=0),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1),
    'Grad Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42),
    'Naive Bayes': GaussianNB()
}

best_models = {}

for name, model in models_7.items():
    model.fit(X_train_under, y_train_under)
    proba_val = model.predict_proba(X_val)[:, 1]

    # 임계값 탐색: proba <= t 인 것만 담았을 때 샤프 최대화
    best_t, best_sharpe = 0.5, -999
    for t in np.quantile(proba_val, np.linspace(0.1, 1.0, 50)):
        mask = proba_val <= t
        if mask.sum() < 500:
            continue
        sh = calc_sharpe(r_val[mask], rf_val[mask])
        if np.isfinite(sh) and sh > best_sharpe:
            best_sharpe = sh
            best_t = t

    approved = proba_val <= best_t
    best_models[name] = {'model': model, 'threshold': best_t, 'sharpe': best_sharpe}

    print(f"[{name:15s}] 샤프: {best_sharpe:7.4f} | "
          f"임계값: {best_t:.3f} | "
          f"승인: {approved.sum():,}건 ({approved.mean()*100:.1f}%) | "
          f"vs 전부대출: {best_sharpe - all_sharpe:+.4f}")

# 최우수 모델
winner = max(best_models, key=lambda k: best_models[k]['sharpe'])
print(f"\n★ 최우수 모델: {winner} (샤프 {best_models[winner]['sharpe']:.4f})")

X_train_under: (246136, 72), NaN: 0
dtypes: {dtype('float64'): 57, dtype('bool'): 8, dtype('int64'): 6, dtype('int32'): 1}
=== 4. 7개 모델 학습 및 샤프 비율 평가 ===
[벤치마크] 전부 대출 시 샤프: -0.0886
국채 금리 범위: 0.36% ~ 4.35%

[LightGBM       ] 샤프:  1.4479 | 임계값: 0.174 | 승인: 150,365건 (70.6%) | vs 전부대출: +1.5365
[XGBoost        ] 샤프:  1.4357 | 임계값: 0.153 | 승인: 150,365건 (70.6%) | vs 전부대출: +1.5243
[CatBoost       ] 샤프:  1.4445 | 임계값: 0.143 | 승인: 146,454건 (68.8%) | vs 전부대출: +1.5331
[Random Forest  ] 샤프:  0.6526 | 임계값: 0.325 | 승인: 21,295건 (10.0%) | vs 전부대출: +0.7411
[Grad Boosting  ] 샤프:  1.4523 | 임계값: 0.168 | 승인: 146,454건 (68.8%) | vs 전부대출: +1.5409
[AdaBoost       ] 샤프:  1.3950 | 임계값: 0.467 | 승인: 119,178건 (56.0%) | vs 전부대출: +1.4836
[Naive Bayes    ] 샤프:  0.1108 | 임계값: 0.008 | 승인: 36,940건 (17.3%) | vs 전부대출: +0.1993

★ 최우수 모델: Grad Boosting (샤프 1.4523)
